# Validation — Raw data (full network, B2B + B2C)

## Kế hoạch validate

**Mục tiêu:** hiểu data gốc trước khi chọn quy tắc cleaning. Notebook này chỉ đọc và kiểm tra: không xóa dòng nào, không ghi file nào. Scope là **toàn mạng**, không loại khách hàng nào; đảo, quốc gia và kênh B2B/B2C chỉ dùng để breakdown.

**Hai nhóm data, tách riêng:**
- **Master + Sales order + GF / Unconstrained** (`3_GF_Unconstrained/`, `ENO_CostLastMileB2B_Unconstaint`): data dùng cho model. Flow tự do, không bị ép theo route hiện tại.
- **Baseline** (`0,1_Baseline2025/`): **forced flow**, mô phỏng lại mạng hiện tại (route, share bị ép theo lịch sử). Chỉ dùng để tham chiếu / đối chiếu với GF, không đưa thẳng vào model.

**Thứ tự:**
1. **Load data**: master + sales order, GF / Unconstrained, Baseline (riêng).
2. **Inspect** từng bảng của nhóm model. Với mỗi bảng kiểm tra:
   - Shape, kiểu dữ liệu, null.
   - Key có unique không.
   - Giá trị bất thường (âm, bằng 0, quá lớn, sai đơn vị).
   - Key có khớp với các bảng khác không (orphan).
   - Breakdown theo đảo / quốc gia / kênh.
3. **Baseline (forced flow)**: inspect riêng, và so sánh với GF (lane nào chỉ có ở một bên, cost lệch bao nhiêu).
4. **Tổng kết**: một bảng các vấn đề tìm được (bảng nào, lỗi gì, bao nhiêu dòng, bao nhiêu kg), và kích thước model theo từng phương án scope. Đưa kết quả vào `Decision_Log.md` §5.

**Câu hỏi chính cần trả lời:**
- Dòng `OrderStatus = Canceled` là soạn hàng lại (đã có bản Done) hay hủy hẳn? Demand nên tính theo `Ordered` hay `Delivered`?
- `SOLineID` có bị lặp hoặc tái sử dụng cho nhiều đơn khác nhau không?
- Mỗi ship-to có thuộc đúng một group không? Group và B2C city nào không có lane last-mile?
- GF có thiếu gì mà chỉ Baseline có không (ví dụ cột cost của Facility Master)?
- Có file lead time chưa, và nó phủ được bao nhiêu cặp facility – demand node?


## 1. Load data

### 1.1 Setup

In [2]:
# ===== SETUP =====
# Import thư viện
import pandas as pd            # đọc / xử lý bảng dữ liệu (DataFrame)
import numpy as np             # tính toán số học (dùng ở phần inspect)
from pathlib import Path       # ghép đường dẫn file, chạy được trên cả Windows lẫn Mac

# Cách pandas hiển thị bảng khi print
pd.set_option("display.width", 200)          # cho phép 1 dòng in rộng 200 ký tự, đỡ bị xuống dòng
pd.set_option("display.max_columns", 40)     # hiện tối đa 40 cột, không bị "..." ở giữa

# Đường dẫn tới data gốc (tương đối so với folder "Coding notebook")
RAW = Path("../Raw Data")                  # folder data gốc
BL = RAW / "0,1_Baseline2025"              # Baseline: forced flow theo mạng hiện tại -> chỉ tham chiếu
GF = RAW / "3_GF_Unconstrained"            # Greenfield Unconstrained: flow tự do -> dùng cho model

### 1.2 Master data + Sales order

In [3]:
# ===== MASTER DATA (không phụ thuộc scenario) =====
# Danh sách khách hàng: 1 dòng = 1 ship-to B2B hoặc 1 thành phố B2C, kèm tọa độ, đảo, quốc gia
customer_master = pd.read_parquet(RAW / "CustomerMaster.parquet")

# Danh sách sản phẩm: 1 dòng = 1 ProductID, kèm segment, trọng lượng, KGPerPallet (để đổi kg -> pallet)
product_master = pd.read_parquet(RAW / "ProductMaster.parquet")

# Bảng phân bổ demand B2B: cho biết mỗi ship-to (ShipToID) thuộc ship-to group nào (ShipToGroupID)
# -> ship-to group chính là demand node trong model
demand_alloc_b2b = pd.read_parquet(RAW / "ENO_DemandAllocationB2B.parquet")

In [4]:
# ===== SALES ORDER =====
# File B2B rất lớn (~2GB, 25 triệu dòng, 27 cột) -> chỉ đọc 10 cột cần dùng cho validate để đỡ RAM
SO_COLS = [
    "SOLineID",          # mã dòng đơn hàng
    "PickingID",         # mã phiếu soạn hàng (1 dòng đơn có thể có nhiều phiếu)
    "OrderStatus",       # trạng thái: Done / Canceled
    "OrderedTimestamp",  # thời điểm đặt hàng -> dùng để chia theo tháng
    "ProductID",         # sản phẩm -> nối với product_master
    "FacilityID",        # kho xuất hàng
    "ShipToID",          # khách nhận hàng -> nối với customer_master / demand_alloc_b2b
    "Country",           # Indonesia / Malaysia
    "QtyOrderedInKG",    # lượng đặt (kg)
    "QtyDeliveredInKG",  # lượng thực giao (kg)
]

# Các cột chữ lặp lại nhiều lần -> đổi sang kiểu "category" (lưu mỗi giá trị 1 lần) để giảm RAM
CAT_COLS = ["OrderStatus", "ProductID", "FacilityID", "ShipToID", "Country"]

def load_so(path):
    """Đọc 1 file sales order: chỉ lấy SO_COLS, rồi đổi các cột CAT_COLS sang category."""
    df = pd.read_parquet(path, columns=SO_COLS)   # đọc đúng các cột cần
    for c in CAT_COLS:                            # duyệt từng cột chữ
        df[c] = df[c].astype("category")          # đổi kiểu -> giảm RAM đáng kể
    return df                                     # trả về bảng đã load

# Đọc 2 file sales order (B2B và B2C có cùng cấu trúc cột)
so_b2b = load_so(RAW / "B2B_SO_2025_JantoNov.parquet")   # B2B, Jan–Nov 2025 (cell nặng nhất, chờ vài phút)
so_b2c = load_so(RAW / "B2C_SO_2025_JantoNov.parquet")   # B2C (e-commerce), Jan–Nov 2025

### 1.3 GF / Unconstrained (dùng cho model)

In [5]:
# ===== GF / UNCONSTRAINED (data dùng cho model) =====
# Danh sách facility (kho) bản GF: gồm kho hiện có + các site greenfield "Non-existent" (chưa tồn tại)
facility_master_gf = pd.read_parquet(GF / "ENO_FacilityMaster.parquet")

# Phân bổ demand bản GF: B2B (ship-to -> group) và B2C (thành phố B2C)
demand_alloc_b2b_gf = pd.read_parquet(GF / "ENO_DemandAllocationB2B.parquet")
demand_alloc_b2c_gf = pd.read_parquet(GF / "ENO_DemandAllocationB2C.parquet")

# Bảng cost theo lane (Origin -> Destination):
cost_supply_gf = pd.read_parquet(GF / "ENO_CostSupply.parquet")              # Nhà máy -> NDC
cost_transfer_gf = pd.read_parquet(GF / "ENO_CostTransfer.parquet")          # Kho -> kho (mid-mile)
cost_lastmile_b2b_gf = pd.read_parquet(GF / "ENO_CostLastMileB2B.parquet")   # Kho -> ship-to group B2B
cost_lastmile_b2b_unc = pd.read_parquet(RAW / "ENO_CostLastMileB2B_Unconstaint.parquet")  # bản Unconstrained của last-mile B2B (nhiều lane hơn)
cost_lastmile_b2c_gf = pd.read_parquet(GF / "ENO_CostLastMileB2C.parquet")   # Kho -> thành phố B2C (cost theo kg)

# DOS (Days of Supply) theo kho × nhóm sản phẩm -> dùng đổi capacity tồn kho thành throughput
dos_gf = pd.read_parquet(GF / "ENO_DOS.parquet")

# Lead time + khoảng cách cho mọi lane (FlowType: NDC/DC/FC/MFC -> DC/Customer/B2C City)
# Cột chính: TotalLeadTimeInDays (cho f2) và DistanceInKM (cho f3 - CO2)
lead_time = pd.read_parquet(RAW / "LeadTime_AllFlow.parquet")

### 1.4 Baseline — forced flow (chỉ tham chiếu)

In [6]:
# ===== BASELINE (forced flow theo mạng hiện tại -> chỉ để so sánh với GF) =====
# Facility master bản Baseline: chỉ kho đang hoạt động, nhưng có thêm cột cost
# (FixedStorageCost_perMonth, MonthsPresentIn2025) mà bản GF không có
facility_master_bl = pd.read_parquet(BL / "ENO_FacilityMaster.parquet")

# Phân bổ demand B2C bản Baseline (đối chiếu với bản GF)
demand_alloc_b2c_bl = pd.read_parquet(BL / "ENO_DemandAllocationB2C.parquet")

# Các bảng cost bản Baseline: chỉ có các lane mạng hiện tại đang chạy
cost_supply_bl = pd.read_parquet(BL / "ENO_CostSupply.parquet")              # Nhà máy -> NDC
cost_transfer_bl = pd.read_parquet(BL / "ENO_CostTransfer.parquet")          # Kho -> kho
cost_lastmile_b2b_bl = pd.read_parquet(BL / "ENO_CostLastMileB2B.parquet")   # Kho -> ship-to group B2B
cost_lastmile_b2c_bl = pd.read_parquet(BL / "ENO_CostLastMileB2C.parquet")   # Kho -> thành phố B2C

### 1.5 Kiểm tra đã load đủ

In [7]:
# ===== KIỂM TRA ĐÃ LOAD ĐỦ =====
# Gom tên các bảng đã load theo 3 nhóm, để in 1 bảng tóm tắt
groups = {
    "Master + SO": ["customer_master", "product_master", "demand_alloc_b2b", "so_b2b", "so_b2c"],
    "GF / Unconstrained": ["facility_master_gf", "demand_alloc_b2b_gf", "demand_alloc_b2c_gf",
                           "cost_supply_gf", "cost_transfer_gf", "cost_lastmile_b2b_gf",
                           "cost_lastmile_b2b_unc", "cost_lastmile_b2c_gf", "dos_gf", "lead_time"],
    "Baseline (forced)": ["facility_master_bl", "demand_alloc_b2c_bl", "cost_supply_bl",
                          "cost_transfer_bl", "cost_lastmile_b2b_bl", "cost_lastmile_b2c_bl"],
}

rows = []                                    # mỗi phần tử = 1 dòng của bảng tóm tắt
for group, names in groups.items():          # duyệt từng nhóm
    for n in names:                          # duyệt từng tên bảng trong nhóm
        df = globals()[n]                    # lấy DataFrame theo tên biến (vd "so_b2b" -> so_b2b)
        rows.append({
            "group": group,                                               # thuộc nhóm nào
            "table": n,                                                   # tên bảng
            "rows": len(df),                                              # số dòng
            "cols": df.shape[1],                                          # số cột
            "RAM_MB": round(df.memory_usage(deep=True).sum() / 1e6, 1),   # dung lượng RAM (MB)
        })

# Hiển thị bảng tóm tắt, index 2 tầng: nhóm -> tên bảng
pd.DataFrame(rows).set_index(["group", "table"])

rows  cols  RAM_MB
group              table                                        
Master + SO        customer_master           48001    16    11.9
                   product_master             5174    31     2.3
                   demand_alloc_b2b        5835102     6   665.8
                   so_b2b                 25260049    10  1648.5
                   so_b2c                  1626339    10   103.6
GF / Unconstrained facility_master_gf          139    18     0.0
                   demand_alloc_b2b_gf     5027090     6   575.2
                   demand_alloc_b2c_gf      559581     5    43.9
                   cost_supply_gf                8     3     0.0
                   cost_transfer_gf           3000     5     0.1
                   cost_lastmile_b2b_gf      16822     3     1.2
                   cost_lastmile_b2b_unc     22852     3     1.6
                   cost_lastmile_b2c_gf       1012     5     0.1
                   dos_gf                    19738     3     0.8
                   lead_time                293161    33   112.4
Baseline (forced)  facility_master_bl           67    20     0.0
                   demand_alloc_b2c_bl      572689     5    44.9
                   cost_supply_bl                4     3     0.0
                   cost_transfer_bl             95     3     0.0
                   cost_lastmile_b2b_bl       3635     3     0.2
                   cost_lastmile_b2c_bl       9122     5     0.7

## 2. Inspect — Master + GF / Unconstrained

### 2.1 Customer Master

In [8]:
# ===== 2.1 CUSTOMER MASTER =====
cm = customer_master                                          # đặt tên ngắn cho dễ viết

# ---------- (1) CẤU TRÚC ----------
print("Shape:", cm.shape)                                     # số dòng, số cột
print(cm.dtypes.to_string())                                  # kiểu dữ liệu từng cột
print("\nÔ trống mỗi cột:", cm.isna().sum()[lambda s: s > 0].to_dict())   # chỉ in cột có ô trống
print("LocationID bị trùng:", cm["LocationID"].duplicated().sum())        # kỳ vọng = 0

# ---------- (2) PHÂN LOẠI ----------
print("\nType × LocationType:")                               # B2B phải đi với ShipToID, B2C đi với B2CCity
print(pd.crosstab(cm["Type"], cm["LocationType"]))
print("\nType × CustomerType:")                               # 2 cột này có nói cùng một điều không
print(pd.crosstab(cm["Type"], cm["CustomerType"]))
print("\nĐảo × Type:")                                         # số khách mỗi đảo, B2B và B2C
print(pd.crosstab(cm["MainIsland"], cm["Type"], margins=True))
print("\nĐảo × Quốc gia:")                                     # đảo Malaysia phải đi với Country = Malaysia
print(pd.crosstab(cm["MainIsland"], cm["Country"]))
print("\nIsJabodetabek theo đảo:")                             # Jabodetabek (vùng Jakarta) chỉ được nằm ở Java
print(pd.crosstab(cm["MainIsland"], cm["IsJabodetabek"]))

# ---------- (3) TỌA ĐỘ ----------
no_coord = cm[cm[["CentroidLat", "CentroidLong"]].isna().any(axis=1)]          # thiếu tọa độ
zero_coord = cm[(cm["CentroidLat"] == 0) | (cm["CentroidLong"] == 0)]          # tọa độ = 0 (lỗi geocode)
print(f"\nThiếu tọa độ: {len(no_coord)} | Tọa độ = 0: {len(zero_coord)}")
print(no_coord[["LocationID", "Type", "District", "City", "MainIsland"]])

# Khung tọa độ gần đúng của từng đảo: (vĩ độ min, vĩ độ max, kinh độ min, kinh độ max)
BBOX = {
    "Java": (-9.0, -5.0, 105.0, 116.0),       "Sumatra": (-6.5, 6.5, 95.0, 109.0),
    "Kalimantan": (-4.5, 4.7, 108.5, 119.5),  "Sulawesi": (-7.5, 5.5, 118.0, 127.0),
    "Bali - Nusra": (-11.2, -7.7, 114.3, 125.5), "Maluku": (-8.6, 3.0, 124.0, 135.0),
    "Papua": (-9.5, 1.0, 129.0, 141.5),       "Peninsular Malaysia": (1.0, 7.0, 99.5, 104.7),
    "East Malaysia": (0.8, 7.5, 109.5, 119.5),
}
def ngoai_dao(r):
    """True nếu tọa độ nằm ngoài khung của đảo khai báo (nghi geocode sai)."""
    b = BBOX.get(r["MainIsland"])                             # khung của đảo khách đó
    if b is None or pd.isna(r["CentroidLat"]):                # không có khung / không có tọa độ -> bỏ qua
        return False
    return not (b[0] <= r["CentroidLat"] <= b[1] and b[2] <= r["CentroidLong"] <= b[3])

sai_dao = cm[cm.apply(ngoai_dao, axis=1)]                     # áp cho từng dòng
print(f"\nTọa độ nằm ngoài đảo khai báo: {len(sai_dao)}")
print(sai_dao[["LocationID", "Type", "District", "City", "MainIsland", "CentroidLat", "CentroidLong"]].to_string())


Shape: (48001, 16)
LocationID           str
Type                 str
LocationType         str
CentroidLong     float64
CentroidLat      float64
Channel              str
SubChannel           str
ChannelName          str
CustomerType         str
SalesAreas           str
District             str
City                 str
Province             str
MainIsland           str
Country              str
IsJabodetabek       bool

Ô trống mỗi cột: {'CentroidLong': 3, 'CentroidLat': 3, 'SalesAreas': 1, 'District': 649}
LocationID bị trùng: 0

Type × LocationType:
LocationType  B2CCity  ShipToID
Type                           
B2B                 0     47352
B2C               649         0

Type × CustomerType:
CustomerType    B2B  B2B2C  B2C
Type                           
B2B           46486    864    2
B2C               0      0  649

Đảo × Type:
Type                   B2B  B2C    All
MainIsland                            
Bali - Nusra          2687   41   2728
East Malaysia          214   53    267

In [9]:
# ===== 2.1d XUẤT CUSTOMER MASTER (đã validate, không cần sửa) =====
OUT = Path("../Model - Data used")                     # folder chứa data đã clean, Model.ipynb đọc từ đây
OUT.mkdir(exist_ok=True)                               # tạo folder nếu chưa có

# Giữ nguyên toàn bộ 48.001 dòng (B2B + B2C chưa chốt). Không sửa tọa độ: 28 khách sai đảo có 0 tấn demand
out_file = OUT / "customer_master.parquet"
customer_master.to_parquet(out_file, index=False)      # index=False: không lưu cột số thứ tự của pandas

# Đọc lại để chắc file ghi đúng
check = pd.read_parquet(out_file)
print(f"Đã lưu: {out_file} | {check.shape[0]:,} dòng × {check.shape[1]} cột")
assert check.shape == customer_master.shape            # báo lỗi nếu số dòng/cột không khớp
assert check["LocationID"].is_unique                    # báo lỗi nếu mã khách bị trùng


Đã lưu: ..\Model - Data used\customer_master.parquet | 48,001 dòng × 16 cột


### 2.2 Product Master

In [10]:
# ===== 2.2a CẤU TRÚC =====
pm = product_master                                                    # tên ngắn

print("Shape:", pm.shape)                                              # số dòng, số cột
print("ProductID bị trùng:", pm["ProductID"].duplicated().sum())       # kỳ vọng = 0
print("Ô trống:", pm.isna().sum()[lambda s: s > 0].to_dict())         # chỉ in cột có ô trống

Shape: (5174, 31)
ProductID bị trùng: 0
Ô trống: {'CBM': 940, 'UnitBoxLengthMm': 937, 'UnitBoxWidthMm': 937, 'UnitBoxHeightMm': 932, 'InnerBoxLengthMm': 3626, 'InnerBoxWidthMm': 3628, 'InnerBoxHeightMm': 3629, 'MasterBoxLengthMm': 1015, 'MasterBoxWidthMm': 1015, 'MasterBoxHeightMm': 1015, 'FactoryID': 170, 'Segment': 4, 'ProductDescription': 293, 'ContainerType': 19, 'PTG': 170, 'PTGCode': 170, 'FactoryType': 170}


In [11]:
# ===== 2.2b SEGMENT =====
# Nhóm sản phẩm: Beauty / Personal Care / Lifestyle / Health & Wellness / trống
print("Số sản phẩm theo Segment:", pm["Segment"].value_counts(dropna=False).to_dict())  # dropna=False: đếm cả ô trống

Số sản phẩm theo Segment: {'Beauty': 4581, 'Personal Care': 376, 'Lifestyle': 186, 'Health & Wellness': 27, nan: 4}


In [12]:
# ===== 2.2c KGPerPallet: phân phối, giá trị vô lý, đối chiếu với cách tính lại =====
# KGPerPallet dùng để đổi kg -> pallet (đơn vị của model) nên phải đúng
print("KGPerPallet:")
print(pm["KGPerPallet"].describe(percentiles=[.01, .05, .5, .95, .99]).round(1))   # min, max, các phân vị

# Gắn cờ giá trị vô lý: < 10 kg/pallet (quá nhẹ) hoặc > 2000 kg/pallet (nặng hơn 1 pallet thực tế)
pm["KGPP_flag"] = np.select([pm["KGPerPallet"] < 10, pm["KGPerPallet"] > 2000],
                            ["<10", ">2000"], default="OK")
print("\nCờ KGPerPallet × Segment:")
print(pd.crosstab(pm["Segment"].fillna("NaN"), pm["KGPP_flag"], margins=True))

# Tính lại KGPerPallet từ 2 cột khác: trọng lượng 1 cái (gram) × số cái / pallet ÷ 1000
pm["KGPP_tinh_lai"] = pm["GrossWeightInGram"] * pm["QuantityPiecesPerPallet"] / 1000
pm["lech_lan"] = pm["KGPerPallet"] / pm["KGPP_tinh_lai"]                           # = 1 nếu 2 cách khớp nhau
print("\nTỷ lệ KGPerPallet / (GrossWeight × PiecesPerPallet):")
print(pm["lech_lan"].describe(percentiles=[.05, .5, .95]).round(2))
print("Sản phẩm lệch > 10%:", ((pm["lech_lan"] - 1).abs() > 0.1).sum())

# Xem vài sản phẩm bị gắn cờ: nếu KGPP_tinh_lai hợp lý thì có thể SỬA thay vì bỏ
print(pm.loc[pm["KGPP_flag"] != "OK",
             ["ProductID", "Segment", "GrossWeightInGram", "QuantityPiecesPerPallet",
              "KGPerPallet", "KGPP_tinh_lai"]].head(10).to_string())

KGPerPallet:
count      5174.0
mean        782.2
std        5770.4
min           0.1
1%            1.0
5%            7.9
50%         236.9
95%        1260.0
99%       10000.0
max      212000.0
Name: KGPerPallet, dtype: float64

Cờ KGPerPallet × Segment:
KGPP_flag          <10  >2000    OK   All
Segment                                  
Beauty             126    113  4342  4581
Health & Wellness    0      4    23    27
Lifestyle          166      0    20   186
NaN                  4      0     0     4
Personal Care        6     95   275   376
All                302    212  4660  5174

Tỷ lệ KGPerPallet / (GrossWeight × PiecesPerPallet):
count    5174.0
mean        1.0
std         0.0
min         1.0
5%          1.0
50%         1.0
95%         1.0
max         1.0
Name: lech_lan, dtype: float64
Sản phẩm lệch > 10%: 0
    ProductID Segment  GrossWeightInGram  QuantityPiecesPerPallet  KGPerPallet  KGPP_tinh_lai
11      00053  Beauty               0.70                   5040.0      3.52800  

In [13]:
# ===== 2.2d KHỚP VỚI SALES ORDER B2B =====
done = so_b2b[so_b2b["OrderStatus"] == "Done"]                    # chỉ đơn đã giao
tong = done["QtyDeliveredInKG"].sum()                             # tổng kg B2B đã giao
pid = done["ProductID"].astype(str)                               # đổi category -> chữ để tra cứu

khong_co = ~pid.isin(pm["ProductID"])                             # True = sản phẩm có đơn nhưng không có trong master
print(f"Sản phẩm có đơn nhưng KHÔNG có trong master: {pid[khong_co].nunique()} mã | "
      f"{done.loc[khong_co.values, 'QtyDeliveredInKG'].sum() / tong:.2%} tổng kg")

Sản phẩm có đơn nhưng KHÔNG có trong master: 0 mã | 0.00% tổng kg


In [14]:
# ===== 2.2e MỨC ẢNH HƯỞNG: % demand B2B theo Segment × cờ KGPerPallet =====
# Dùng để quyết định: bỏ segment nào, bỏ hay sửa sản phẩm lỗi KGPerPallet
seg = pid.map(pm.set_index("ProductID")["Segment"]).fillna("không có/NaN")    # segment của từng dòng đơn
flag = pid.map(pm.set_index("ProductID")["KGPP_flag"]).fillna("không có")     # cờ KGPerPallet của từng dòng đơn

anh_huong = (pd.DataFrame({"Segment": seg.values, "KGPP": flag.values, "kg": done["QtyDeliveredInKG"].values})
             .groupby(["Segment", "KGPP"])["kg"].sum() / tong * 100)          # % tổng kg
print("% tổng kg B2B theo Segment × cờ KGPerPallet:")
print(anh_huong.unstack(fill_value=0).round(2))                                # Segment theo dòng, cờ theo cột

% tổng kg B2B theo Segment × cờ KGPerPallet:
KGPP                <10  >2000     OK
Segment                              
Beauty             0.38   0.33  76.19
Health & Wellness  0.00   0.00   0.00
Lifestyle          0.00   0.00   0.00
Personal Care      0.00   0.50  22.60
không có/NaN       0.00   0.00   0.00


In [15]:
# ===== 2.2f CLEAN + XUẤT PRODUCT MASTER =====
# Quyết định (Decision_Log 5.4): bỏ sản phẩm có KGPerPallet < 10 hoặc > 2000 (lỗi dữ liệu, 1.21% kg B2B)
# Không lọc theo Segment: Lifestyle / Health & Wellness / trống có 0% kg B2B, tự rơi ra khi gom demand

LOW_KGPP, HIGH_KGPP = 10, 2000                                         # ngưỡng kg/pallet hợp lý

# Bỏ các cột phụ đã thêm lúc inspect (pm là cùng 1 bảng với product_master nên chúng đã bị thêm vào)
pm_clean = product_master.drop(columns=["KGPP_flag", "KGPP_tinh_lai", "lech_lan"], errors="ignore")

truoc = len(pm_clean)                                                   # số sản phẩm trước khi lọc
pm_clean = pm_clean[pm_clean["KGPerPallet"].between(LOW_KGPP, HIGH_KGPP)].copy()   # giữ 10 <= KGPerPallet <= 2000
print(f"Product master: {truoc:,} -> {len(pm_clean):,} sản phẩm (bỏ {truoc - len(pm_clean):,})")

# Kiểm tra tự động trước khi xuất: sai thì cell báo lỗi, không ghi file
assert pm_clean["ProductID"].is_unique                                  # không trùng mã
assert pm_clean["KGPerPallet"].notna().all()                            # không trống KGPerPallet
assert pm_clean["KGPerPallet"].between(LOW_KGPP, HIGH_KGPP).all()       # không còn giá trị vô lý

# Xuất ra folder data đã clean
OUT = Path("../Model - Data used")
OUT.mkdir(exist_ok=True)
out_file = OUT / "product_master.parquet"
pm_clean.to_parquet(out_file, index=False)                              # index=False: không lưu cột số thứ tự

# Đọc lại để chắc file ghi đúng
check = pd.read_parquet(out_file)
assert check.shape == pm_clean.shape                                    # số dòng / cột khớp
print(f"Đã lưu: {out_file} | {check.shape[0]:,} dòng × {check.shape[1]} cột")

Product master: 5,174 -> 4,660 sản phẩm (bỏ 514)
Đã lưu: ..\Model - Data used\product_master.parquet | 4,660 dòng × 31 cột


### 2.3 Facility Master (GF)

In [16]:
# ===== 2.3a TỔNG QUAN KHO HIỆN CÓ (Facility Master BL) =====
# Theo Decision_Log 4.2 / 4.4a: danh sách kho, capacity và fixed cost lấy từ BL (mạng hiện tại 2025)
fm = facility_master_bl                                                  # tên ngắn

print("Shape:", fm.shape)
print("LocationID bị trùng:", fm["LocationID"].duplicated().sum())      # kỳ vọng = 0
print("Status:", fm["Status"].value_counts().to_dict())                 # kỳ vọng toàn Active

print("\nLoại kho × Đảo:")
print(pd.crosstab(fm["FacilityType"], fm["MainIsland"], margins=True))


Shape: (67, 20)
LocationID bị trùng: 0
Status: {'Active': 67}

Loại kho × Đảo:
MainIsland    Bali - Nusra  Java  Kalimantan  Maluku  Peninsular Malaysia  Sulawesi  Sumatra  All
FacilityType                                                                                     
DC Direct                0     5           1       0                    1         0        1    8
DC Satellite             2     3           2       1                    0         3        7   18
DEPO                     0     9           3       1                    0         2        3   18
FC                       1     3           2       0                    0         2        3   11
Instant Hub              0     2           0       0                    0         0        0    2
NDC                      0     1           0       0                    0         0        0    1
RDC                      0     4           1       0                    0         1        3    9
All                      3    27       

In [19]:
# ===== 2.3b CAPACITY + FIXED COST theo loại kho =====
# Số kho có giá trị = 0, và giá trị trung vị, theo từng loại kho
tom_tat = fm.groupby("FacilityType").agg(
    so_kho=("LocationID", "size"),
    cap_bang_0=("CapacityInPallet", lambda s: (s.fillna(0) == 0).sum()),             # kho không có capacity
    cap_trung_vi=("CapacityInPallet", "median"),                                     # pallet
    fixed_bang_0=("FixedStorageCost_perMonth", lambda s: (s.fillna(0) == 0).sum()),  # kho không có fixed cost
    fixed_thang_trung_vi=("FixedStorageCost_perMonth", "median"),                    # IDR / tháng
    handling_trung_vi=("B2BHandlingCostPerPallet", "median"),                        # IDR / pallet
)
print(tom_tat.round(0).to_string())

# Kiểm tra cách tính fixed cost theo tháng: perMonth = FixedStorageCost / số tháng hoạt động trong 2025
tinh_lai = fm["FixedStorageCost"] / fm["MonthsPresentIn2025"]
lech = (tinh_lai / fm["FixedStorageCost_perMonth"] - 1).abs()
print(f"\nperMonth = FixedStorageCost / MonthsPresentIn2025? lệch lớn nhất: {lech.max():.4%}")

# Kho không chạy đủ 12 tháng trong 2025 (mở / đóng giữa năm)
print("\nKho hoạt động < 12 tháng trong 2025:")
print(fm.loc[fm["MonthsPresentIn2025"] < 12,
             ["LocationID", "FacilityName", "FacilityType", "MonthsPresentIn2025"]].to_string(index=False))


              so_kho  cap_bang_0  cap_trung_vi  fixed_bang_0  fixed_thang_trung_vi  handling_trung_vi
FacilityType                                                                                         
DC Direct          8           0         274.0             0           428295548.0           369590.0
DC Satellite      18           0         126.0             0           152228192.0           454443.0
DEPO              18          18           0.0             0            41909653.0           189436.0
FC                11           0          16.0             0            18885609.0           454443.0
Instant Hub        2           0          38.0             0           102821721.0           369590.0
NDC                1           0        6220.0             0           636311012.0           115919.0
RDC                9           0         320.0             0           471360000.0           434575.0

perMonth = FixedStorageCost / MonthsPresentIn2025? lệch lớn nhất: 0.0000%

Kho ho

In [20]:
# ===== 2.3c LOẠI KHO NÀO PHỤC VỤ B2B? (để quyết định giữ / bỏ FC, Instant Hub) =====
# Cách 1: kho nào có tuyến giao hàng B2B (bảng giá last-mile B2B, gộp 2 file GF + Unc)
lm_b2b = pd.concat([cost_lastmile_b2b_gf, cost_lastmile_b2b_unc]).drop_duplicates(["OriginID", "DestinationID"])
so_tuyen = lm_b2b.groupby("OriginID").size().rename("so_tuyen_B2B")             # số tuyến B2B mỗi kho
fm_b2b = fm.set_index("LocationID").join(so_tuyen).fillna({"so_tuyen_B2B": 0})

# Cách 2: trong thực tế 2025, kho nào đã xuất hàng B2B (đơn Done) và bao nhiêu tấn
done = so_b2b[so_b2b["OrderStatus"] == "Done"]
tan = done.groupby("FacilityID", observed=True)["QtyDeliveredInKG"].sum().div(1000).rename("tan_B2B_2025")
fm_b2b = fm_b2b.join(tan).fillna({"tan_B2B_2025": 0})

print("Theo loại kho: số kho / số kho có tuyến B2B / số kho có xuất B2B 2025 / tấn B2B 2025")
print(fm_b2b.groupby("FacilityType").agg(
    so_kho=("FacilityName", "size"),
    co_tuyen_B2B=("so_tuyen_B2B", lambda s: (s > 0).sum()),
    co_xuat_B2B=("tan_B2B_2025", lambda s: (s > 0).sum()),
    tan_B2B=("tan_B2B_2025", "sum")).round(0).to_string())

# Chi tiết FC / Instant Hub
print("\nFC / Instant Hub:")
print(fm_b2b[fm_b2b["FacilityType"].isin(["FC", "Instant Hub"])]
      [["FacilityName", "FacilityType", "MainIsland", "so_tuyen_B2B", "tan_B2B_2025"]].round(1).to_string())

# Kho có xuất B2B năm 2025 nhưng KHÔNG có trong facility master BL
la = tan[~tan.index.isin(fm["LocationID"])]
print(f"\nFacilityID xuất B2B nhưng không có trong master: {len(la)} | {la.sum():,.1f} tấn")
print(la.sort_values(ascending=False).head(10).round(1))


Theo loại kho: số kho / số kho có tuyến B2B / số kho có xuất B2B 2025 / tấn B2B 2025
              so_kho  co_tuyen_B2B  co_xuat_B2B  tan_B2B
FacilityType                                            
DC Direct          8             7            4   6041.0
DC Satellite      18            18            4   1519.0
DEPO              18            18            0      0.0
FC                11             0            0      0.0
Instant Hub        2             0            0      0.0
NDC                1             1            1   3135.0
RDC                9             9            5   2895.0

FC / Instant Hub:
                     FacilityName FacilityType    MainIsland  so_tuyen_B2B  tan_B2B_2025
LocationID                                                                              
D42                      FC Tegal           FC          Java           0.0           0.0
D45                     FC Kediri           FC          Java           0.0           0.0
D51                  FC Pal

In [22]:
# ===== 2.3d KHO MALAYSIA: MWH_M02 (BL) -> MWH_M04 (GF) =====
# Notion: Klang (M02) đóng, thay bằng Shah Alam (M04). Kiểm tra các bảng giá đang dùng mã nào
for ma in ["MWH_M02", "MWH_M04"]:
    print(f"{ma}: trong BL={ma in set(fm['LocationID'])} | trong GF={ma in set(facility_master_gf['LocationID'])} | "
          f"tuyến last-mile B2B={(lm_b2b['OriginID'] == ma).sum()} | "
          f"tuyến transfer GF (đi/đến)={cost_transfer_gf['OriginID'].eq(ma).sum()}/{cost_transfer_gf['DestinationID'].eq(ma).sum()}")
print(facility_master_gf.loc[facility_master_gf["LocationID"] == "MWH_M04",
                             ["LocationID", "FacilityName", "FacilityType", "MainIsland", "Latitude", "Longitude"]])


MWH_M02: trong BL=True | trong GF=False | tuyến last-mile B2B=0 | tuyến transfer GF (đi/đến)=0/0
MWH_M04: trong BL=False | trong GF=True | tuyến last-mile B2B=86 | tuyến transfer GF (đi/đến)=4/6
   LocationID  FacilityName FacilityType           MainIsland  Latitude  Longitude
40    MWH_M04  DC Shah Alam    DC Direct  Peninsular Malaysia       3.0      102.0


In [23]:
# ===== 2.3e CHI PHÍ XỬ LÝ HÀNG (handling) BL vs GF: GF có bị nâng giá lên 2030 không? =====
# Nếu GF / BL ≈ 1 -> cùng giá 2025;  > 1 -> GF đã cộng trượt giá, cần quy về 2025 như cost vận chuyển
cot = ["B2BHandlingCostPerPallet", "StoragePenaltyPerPallet"]
so = fm[["LocationID"] + cot].merge(facility_master_gf[["LocationID"] + cot], on="LocationID", suffixes=("_BL", "_GF"))
for c in cot:
    ty_le = so[f"{c}_GF"] / so[f"{c}_BL"]                                          # GF chia BL, từng kho
    print(f"{c}: tỷ lệ GF/BL trung vị = {ty_le.median():.3f} | 5%–95%: {ty_le.quantile(.05):.3f}–{ty_le.quantile(.95):.3f}")



B2BHandlingCostPerPallet: tỷ lệ GF/BL trung vị = 1.407 | 5%–95%: 1.391–1.433
StoragePenaltyPerPallet: tỷ lệ GF/BL trung vị = 0.136 | 5%–95%: 0.136–nan


d:\Python Downloading\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


In [26]:
# ===== 2.3g CLEAN + XUẤT FACILITY MASTER =====
# Quyết định (Decision_Log 4.2, 4.2b, 4.4a, 3.1):
#   - lấy kho + sức chứa + tiền thuê + chi phí xử lý từ BL (giá 2025)
#   - bỏ 13 kho online (FC, Instant Hub) -> còn 54 kho
#   - đổi mã kho Malaysia MWH_M02 -> MWH_M04, lấy tọa độ M04 từ GF
#   - NDC luôn mở (FIXED), 53 kho còn lại để model chọn mở / đóng (DECISION)

fac = facility_master_bl[~facility_master_bl["FacilityType"].isin(["FC", "Instant Hub"])].copy()   # bỏ kho online
print("Số kho sau khi bỏ FC / Instant Hub:", len(fac))

# Đổi mã kho Malaysia: M02 (Klang, mã cũ) -> M04 (Shah Alam, mã bảng giá đang dùng)
m04 = facility_master_gf.set_index("LocationID").loc["MWH_M04"]                    # thông tin M04 trong GF
dong_m02 = fac["LocationID"] == "MWH_M02"                                          # dòng cần đổi
fac.loc[dong_m02, ["LocationID", "FacilityName", "Latitude", "Longitude"]] = \
    ["MWH_M04", m04["FacilityName"], m04["Latitude"], m04["Longitude"]]           # giữ sức chứa + chi phí của BL

# Vai trò trong model
fac["Role"] = np.where(fac["FacilityType"] == "NDC", "FIXED", "DECISION")         # NDC luôn mở
print("Vai trò:", fac["Role"].value_counts().to_dict())                            # kỳ vọng FIXED 1, DECISION 53


Số kho sau khi bỏ FC / Instant Hub: 54
Vai trò: {'DECISION': 53, 'FIXED': 1}


In [27]:
# Kiểm tra từng điều kiện, in ra kết quả thay vì dừng ngay ở lỗi đầu tiên
cot = ["CapacityInPallet", "FixedStorageCost_perMonth", "B2BHandlingCostPerPallet", "Latitude", "Longitude"]
kiem_tra = {
    "Đúng 54 kho": len(fac) == 54,
    "Không trùng mã": fac["LocationID"].is_unique,
    "Đã đổi M02 -> M04": ("MWH_M02" not in set(fac["LocationID"])) and ("MWH_M04" in set(fac["LocationID"])),
    "Không ô trống": bool(fac[cot].notna().all().all()),
    "Mọi kho có tiền thuê": bool((fac["FixedStorageCost_perMonth"] > 0).all()),
    "Chỉ DEPO có sức chứa 0": bool(((fac["CapacityInPallet"] > 0) | (fac["FacilityType"] == "DEPO")).all()),
}
for ten, ok in kiem_tra.items():
    print(f"{'OK  ' if ok else 'LỖI '} {ten}")
print(fac.loc[fac["LocationID"].isin(["MWH_M02", "MWH_M04"]), ["LocationID", "FacilityName"] + cot])  # xem dòng Malaysia

# Chỉ lưu khi tất cả OK
if all(kiem_tra.values()):
    out_file = Path("../Model - Data used") / "facility_master.parquet"
    fac.to_parquet(out_file, index=False)
    check = pd.read_parquet(out_file)
    print(f"\nĐã lưu: {out_file} | {check.shape[0]} kho × {check.shape[1]} cột")
    print(check.groupby(["Role", "FacilityType"]).size())
else:
    print("\nCHƯA LƯU - có kiểm tra bị lỗi, dán kết quả này cho Claude")


OK   Đúng 54 kho
OK   Không trùng mã
OK   Đã đổi M02 -> M04
OK   Không ô trống
OK   Mọi kho có tiền thuê
OK   Chỉ DEPO có sức chứa 0
   LocationID  FacilityName  CapacityInPallet  FixedStorageCost_perMonth  B2BHandlingCostPerPallet  Latitude  Longitude
40    MWH_M04  DC Shah Alam            1200.0               1.017782e+09              4.469409e+06       3.0      102.0

Đã lưu: ..\Model - Data used\facility_master.parquet | 54 kho × 21 cột
Role      FacilityType
DECISION  DC Direct        8
          DC Satellite    18
          DEPO            18
          RDC              9
FIXED     NDC              1
dtype: int64


### 2.4 Demand mapping (B2B ship-to → group, B2C city)

In [28]:
# ===== 2.4a BẢNG GÁN KHÁCH -> NHÓM (ship-to -> ship-to group) =====
# demand_alloc_b2b có 5,8 triệu dòng (khách × sản phẩm) -> chỉ lấy 2 cột rồi bỏ trùng
map_root = demand_alloc_b2b[["ShipToID", "ShipToGroupID"]].drop_duplicates()      # bản ở folder gốc
map_gf = demand_alloc_b2b_gf[["ShipToID", "ShipToGroupID"]].drop_duplicates()     # bản GF

for ten, m in [("Root", map_root), ("GF", map_gf)]:
    nhieu_nhom = (m.groupby("ShipToID")["ShipToGroupID"].nunique() > 1).sum()     # khách thuộc > 1 nhóm
    print(f"{ten}: {m['ShipToID'].nunique():,} khách -> {m['ShipToGroupID'].nunique():,} nhóm | "
          f"khách thuộc >1 nhóm: {nhieu_nhom}")

# Hai bản có gán giống nhau không (trên các khách có ở cả 2)
chung = map_root.merge(map_gf, on="ShipToID", suffixes=("_root", "_gf"))
print(f"\nKhách có ở cả 2 bản: {len(chung):,} | gán khác nhóm: {(chung['ShipToGroupID_root'] != chung['ShipToGroupID_gf']).sum()}")
print(f"Khách chỉ có ở Root: {len(set(map_root['ShipToID']) - set(map_gf['ShipToID'])):,}")
print(f"Khách chỉ có ở GF:   {len(set(map_gf['ShipToID']) - set(map_root['ShipToID'])):,}")


Root: 34,922 khách -> 3,433 nhóm | khách thuộc >1 nhóm: 0
GF: 30,738 khách -> 3,362 nhóm | khách thuộc >1 nhóm: 0

Khách có ở cả 2 bản: 30,738 | gán khác nhóm: 0
Khách chỉ có ở Root: 4,184
Khách chỉ có ở GF:   0


In [29]:
# ===== 2.4b NHÓM THEO ĐẢO + DEMAND MỖI NHÓM =====
m = map_root.merge(customer_master[["LocationID", "MainIsland"]], left_on="ShipToID", right_on="LocationID", how="left")

# Nhóm có khách nằm ở nhiều đảo khác nhau không (nhóm phải gọn trong 1 khu vực)
nhieu_dao = (m.groupby("ShipToGroupID")["MainIsland"].nunique() > 1).sum()
print(f"Nhóm có khách ở >1 đảo: {nhieu_dao}")

# Đảo của mỗi nhóm = đảo xuất hiện nhiều nhất trong nhóm
dao_nhom = m.groupby("ShipToGroupID")["MainIsland"].agg(lambda s: s.mode().iat[0])

# Demand (tấn, đơn Done) của mỗi nhóm
done = so_b2b[so_b2b["OrderStatus"] == "Done"]
kg_khach = done.groupby("ShipToID", observed=True)["QtyDeliveredInKG"].sum()                       # kg theo khách
kg_nhom = m.assign(kg=m["ShipToID"].map(kg_khach).fillna(0)).groupby("ShipToGroupID")["kg"].sum()  # kg theo nhóm

bang = pd.DataFrame({"dao": dao_nhom, "tan": kg_nhom / 1000})
print("\nTheo đảo: số nhóm, số nhóm có demand, tấn")
print(bang.groupby("dao").agg(so_nhom=("tan", "size"),
                              nhom_co_demand=("tan", lambda s: (s > 0).sum()),
                              tan=("tan", "sum")).round(0).to_string())
print(f"\nTổng: {len(bang):,} nhóm | có demand: {(bang['tan'] > 0).sum():,} | {bang['tan'].sum():,.0f} tấn")


Nhóm có khách ở >1 đảo: 0

Theo đảo: số nhóm, số nhóm có demand, tấn
                     so_nhom  nhom_co_demand      tan
dao                                                  
Bali - Nusra             118             118    955.0
East Malaysia             27              27     17.0
Java                    1763            1763  15673.0
Kalimantan               255             255   1526.0
Maluku                    40              40    142.0
Papua                     22              22     74.0
Peninsular Malaysia       59              59    891.0
Sulawesi                 351             351   1468.0
Sumatra                  798             798   3785.0

Tổng: 3,433 nhóm | có demand: 3,433 | 24,532 tấn


In [30]:
# ===== 2.4c NHÓM CÓ DEMAND NHƯNG KHÔNG CÓ KHO NÀO (trong 54 kho) GIAO TỚI =====
kho54 = set(pd.read_parquet(Path("../Model - Data used") / "facility_master.parquet")["LocationID"])   # 54 kho đã chốt
lm = pd.concat([cost_lastmile_b2b_gf, cost_lastmile_b2b_unc]).drop_duplicates(["OriginID", "DestinationID"])  # gộp 2 bảng giá
lm54 = lm[lm["OriginID"].isin(kho54)]                                                  # chỉ tuyến đi từ 54 kho

so_kho = lm54.groupby("DestinationID")["OriginID"].nunique()                           # mỗi nhóm có bao nhiêu kho giao tới
bang["so_kho_giao"] = bang.index.map(so_kho).fillna(0).astype(int)

khong_kho = bang[(bang["tan"] > 0) & (bang["so_kho_giao"] == 0)]
print(f"Nhóm có demand nhưng KHÔNG có kho giao: {len(khong_kho)} nhóm | {khong_kho['tan'].sum():,.1f} tấn "
      f"({khong_kho['tan'].sum() / bang['tan'].sum():.2%})")
print(khong_kho.groupby("dao")["tan"].agg(["size", "sum"]).round(1))

print("\nNhóm có demand: có bao nhiêu kho có thể giao tới (số kho: số nhóm)")
print(bang.loc[bang["tan"] > 0, "so_kho_giao"].value_counts().sort_index().head(10).to_dict())


Nhóm có demand nhưng KHÔNG có kho giao: 0 nhóm | 0.0 tấn (0.00%)
Empty DataFrame
Columns: [size, sum]
Index: []

Nhóm có demand: có bao nhiêu kho có thể giao tới (số kho: số nhóm)
{1: 303, 2: 548, 3: 531, 4: 370, 5: 463, 6: 619, 7: 459, 8: 119, 9: 20, 10: 1}


In [ ]:
# ===== 2.4d TUYẾN GIAO TỚI NHÓM KHÔNG CÓ TRONG BẢNG GÁN =====
# Bảng giá có tuyến tới DestinationID không phải nhóm nào trong bảng gán -> tuyến thừa (hoặc mã lệch)
la = set(lm54["DestinationID"]) - set(bang.index)
print(f"DestinationID tronbbg bảng giá nhưng không phải nhóm nào: {len(la)}")
print(list(la)[:10])


DestinationID trong bảng giá nhưng không phải nhóm nào: 0
[]


In [33]:
# ===== 2.4e XUẤT BẢNG GÁN KHÁCH -> NHÓM =====
# Dùng bản gốc (đủ 34.922 khách); bản GF chỉ là tập con, gán giống hệt
assert map_root["ShipToID"].is_unique                                      # mỗi khách thuộc đúng 1 nhóm
assert map_root["ShipToGroupID"].nunique() == 3433                         # đúng 3.433 nhóm

out_file = Path("../Model - Data used") / "shipto_group_map.parquet"
map_root.to_parquet(out_file, index=False)                                 # lưu 2 cột: ShipToID, ShipToGroupID

check = pd.read_parquet(out_file)                                          # đọc lại để chắc file đúng
assert check.shape == map_root.shape
print(f"Đã lưu: {out_file} | {len(check):,} khách -> {check['ShipToGroupID'].nunique():,} nhóm")


Đã lưu: ..\Model - Data used\shipto_group_map.parquet | 34,922 khách -> 3,433 nhóm


In [32]:
# ===== 2.4f 303 NHÓM CHỈ CÓ 1 KHO GIAO TỚI: ở đâu, kho nào? =====
mot_kho = bang[bang["so_kho_giao"] == 1].copy()                                   # nhóm chỉ có 1 lựa chọn
kho_duy_nhat = lm54.groupby("DestinationID")["OriginID"].first()                  # kho duy nhất đó là kho nào
mot_kho["kho"] = mot_kho.index.map(kho_duy_nhat)

fac54 = pd.read_parquet(Path("../Model - Data used") / "facility_master.parquet").set_index("LocationID")
mot_kho["loai_kho"] = mot_kho["kho"].map(fac54["FacilityType"])                   # loại của kho đó

print(f"{len(mot_kho)} nhóm chỉ có 1 kho | {mot_kho['tan'].sum():,.0f} tấn "
      f"({mot_kho['tan'].sum() / bang['tan'].sum():.1%} tổng demand)")

print("\nTheo đảo (số nhóm, tấn) - so với tổng số nhóm của đảo:")
theo_dao = mot_kho.groupby("dao").agg(nhom_1_kho=("tan", "size"), tan=("tan", "sum"))
theo_dao["tong_nhom_dao"] = bang.groupby("dao").size()
print(theo_dao.round(0).to_string())

print("\nCác kho 'duy nhất' (kho -> số nhóm chỉ trông vào nó):")
print(mot_kho.groupby(["kho", "loai_kho"]).agg(so_nhom=("tan", "size"), tan=("tan", "sum"))
      .sort_values("so_nhom", ascending=False).round(0).to_string())


303 nhóm chỉ có 1 kho | 4,864 tấn (19.8% tổng demand)

Theo đảo (số nhóm, tấn) - so với tổng số nhóm của đảo:
                     nhom_1_kho     tan  tong_nhom_dao
dao                                                   
Bali - Nusra                 17    94.0            118
East Malaysia                27    17.0             27
Java                         37  3236.0           1763
Kalimantan                   25    97.0            255
Maluku                       23    78.0             40
Papua                        17    45.0             22
Peninsular Malaysia          59   891.0             59
Sulawesi                     87   383.0            351
Sumatra                      11    22.0            798

Các kho 'duy nhất' (kho -> số nhóm chỉ trông vào nó):
                      so_nhom     tan
kho     loai_kho                     
MWH_M04 DC Direct          86   908.0
MWH_D40 DC Satellite       56   232.0
MWH_D12 RDC                34   100.0
D44     DC Satellite       24   148.0
MW

In [43]:
# ===== BỎ MALAYSIA KHỎI CÁC FILE ĐÃ XUẤT (Decision_Log 2.1) =====
OUT = Path("../Model - Data used")

# Khách hàng: chỉ giữ Indonesia
cm = pd.read_parquet(OUT / "customer_master.parquet")
cm_id = cm[cm["Country"] == "Indonesia"]
print(f"customer_master: {len(cm):,} -> {len(cm_id):,}")

# Kho: chỉ giữ kho ở Indonesia
fac = pd.read_parquet(OUT / "facility_master.parquet")
fac_id = fac[fac["Country"] == "Indonesia"]
print(f"facility_master: {len(fac)} -> {len(fac_id)} | vai trò: {fac_id['Role'].value_counts().to_dict()}")

# Bảng gán khách -> nhóm: chỉ giữ khách Indonesia
mp = pd.read_parquet(OUT / "shipto_group_map.parquet")
mp_id = mp[mp["ShipToID"].isin(cm_id["LocationID"])]
print(f"shipto_group_map: {len(mp):,} khách / {mp['ShipToGroupID'].nunique():,} nhóm -> "
      f"{len(mp_id):,} khách / {mp_id['ShipToGroupID'].nunique():,} nhóm")

# Kiểm tra rồi lưu đè
assert (fac_id["Country"] == "Indonesia").all() and len(fac_id) == 53            # 53 kho
assert mp_id["ShipToGroupID"].nunique() == 3347                                    # 3.347 nhóm
cm_id.to_parquet(OUT / "customer_master.parquet", index=False)
fac_id.to_parquet(OUT / "facility_master.parquet", index=False)
mp_id.to_parquet(OUT / "shipto_group_map.parquet", index=False)
print("Đã lưu đè 3 file (chỉ Indonesia)")


customer_master: 48,001 -> 46,461
facility_master: 54 -> 53 | vai trò: {'DECISION': 52, 'FIXED': 1}
shipto_group_map: 34,922 khách / 3,433 nhóm -> 34,430 khách / 3,347 nhóm
Đã lưu đè 3 file (chỉ Indonesia)


### 2.5 Sales order B2B

In [34]:
# ===== 2.5a TỔNG QUAN SALES ORDER B2B =====
print("Số dòng:", f"{len(so_b2b):,}")
print("OrderStatus:", so_b2b["OrderStatus"].value_counts().to_dict())          # bao nhiêu dòng Done / Canceled
print("Country:", so_b2b["Country"].value_counts().to_dict())                  # Indonesia / Malaysia
print("Từ", so_b2b["OrderedTimestamp"].min(), "đến", so_b2b["OrderedTimestamp"].max())
print("Ô trống:", so_b2b.isna().sum()[lambda s: s > 0].to_dict() or "không có")
print("Số lượng đặt <= 0:", (so_b2b["QtyOrderedInKG"] <= 0).sum())            # dòng số lượng vô lý
print("Dòng trùng hoàn toàn:", so_b2b.duplicated().sum())                      # mọi cột giống hệt nhau


Số dòng: 25,260,049
OrderStatus: {'Done': 22076254, 'Canceled': 3183795}
Country: {'Indonesia': 24910279, 'Malaysia': 349770}
Từ 2025-01-01 00:00:00+00:00 đến 2025-11-30 15:46:38+00:00
Ô trống: không có
Số lượng đặt <= 0: 1029
Dòng trùng hoàn toàn: 21


In [35]:
# ===== 2.5b CẤU TRÚC SOLineID: 1 dòng đơn có thể có nhiều phiếu soạn hàng =====
n_dong = so_b2b.groupby("SOLineID", observed=True).size()                     # đếm số dòng của mỗi SOLineID
print("Phân phối số dòng / SOLineID:", n_dong.value_counts().sort_index().head(8).to_dict())
print("\nTop 5 SbOLineID nhiều dòng nhất:")
print(n_dong.sort_values(ascending=False).head(5))


Phân phối số dòng / SOLineID: {1: 21467397, 2: 101718, 3: 421, 4: 88, 5: 2, 3587591: 1}

Top 5 SbOLineID nhiều dòng nhất:
SOLineID
NULL         3587591
151355944          5
147662725          5
144235942          4
144235959          4
dtype: int64


In [36]:
# ===== 2.5c CANCELED: đã soạn lại thành Done hay hủy hẳn? =====
KEY = ["SOLineID", "ShipToID", "ProductID"]                                    # khóa nhận diện 1 dòng đơn thật
done_keys = so_b2b.loc[so_b2b["OrderStatus"] == "Done", KEY].drop_duplicates()  # các dòng đơn có bản Done
done_keys["co_ban_Done"] = True
canc = so_b2b[so_b2b["OrderStatus"] == "Canceled"].merge(done_keys, on=KEY, how="left")  # gắn cờ cho dòng Canceled
canc["co_ban_Done"] = canc["co_ban_Done"].fillna(False)

print("Dòng Canceled - có bản Done (soạn lại) hay không (hủy hẳn):")
print(canc.groupby("co_ban_Done")["QtyOrderedInKG"].agg(so_dong="size", tan=lambda s: s.sum() / 1000).round(1))
print("\nTấn đã giao trên dòng Canceled:", round(canc["QtyDeliveredInKG"].sum() / 1000, 1))   # kỳ vọng = 0


Dòng Canceled - có bản Done (soạn lại) hay không (hủy hẳn):
             so_dong     tan
co_ban_Done                 
False        2533394  3454.6
True          650401  1155.6

Tấn đã giao trên dòng Canceled: 0.0


In [37]:
# ===== 2.5d DEMAND THEO 3 CÁCH TÍNH, THEO THÁNG (tấn) =====
thang = so_b2b["OrderedTimestamp"].dt.tz_localize(None).dt.to_period("M")     # cột tháng
done_mask = so_b2b["OrderStatus"] == "Done"

cach = pd.DataFrame({
    "TatCa_Dat": so_b2b.groupby(thang)["QtyOrderedInKG"].sum(),                   # cộng cả Canceled -> có thể trùng
    "Done_Dat": so_b2b[done_mask].groupby(thang[done_mask])["QtyOrderedInKG"].sum(),    # chỉ Done, lượng đặt
    "Done_Giao": so_b2b[done_mask].groupby(thang[done_mask])["QtyDeliveredInKG"].sum(), # chỉ Done, lượng thực giao
}) / 1000
cach.loc["TONG"] = cach.sum()                                                    # dòng tổng
cach["TatCa_vs_DoneGiao_%"] = (cach["TatCa_Dat"] / cach["Done_Giao"] - 1) * 100  # cộng cả Canceled phồng bao nhiêu %
cach["Giao_vs_Dat_%"] = (cach["Done_Giao"] / cach["Done_Dat"]) * 100             # tỷ lệ giao được / đặt (đơn Done)
print(cach.round(1))


                  TatCa_Dat  Done_Dat  Done_Giao  TatCa_vs_DoneGiao_%  Giao_vs_Dat_%
OrderedTimestamp                                                                    
2025-01              2206.0    1925.6     1896.3                 16.3           98.5
2025-02              2212.7    1999.9     1982.8                 11.6           99.1
2025-03              2739.4    2334.7     2303.2                 18.9           98.6
2025-04              2794.7    2160.9     2122.8                 31.7           98.2
2025-05              2902.9    2245.7     2191.3                 32.5           97.6
2025-06              3187.8    2407.6     2350.3                 35.6           97.6
2025-07              3091.6    2528.9     2479.9                 24.7           98.1
2025-08              2827.9    2530.4     2508.9                 12.7           99.2
2025-09              2620.8    2384.5     2363.8                 10.9           99.1
2025-10              2462.0    2240.4     2221.1                 

In [40]:
# ===== 2.5e DÒNG CÓ SOLineID = "NULL": là đơn gì? =====
null_mask = so_b2b["SOLineID"] == "NULL"                                        # dòng không có mã dòng đơn
print("Theo trạng thái:", so_b2b.loc[null_mask, "OrderStatus"].value_counts().to_dict())
print("Tấn đặt / tấn giao:",
      round(so_b2b.loc[null_mask, "QtyOrderedInKG"].sum() / 1000, 1), "/",
      round(so_b2b.loc[null_mask, "QtyDeliveredInKG"].sum() / 1000, 1))
print("Theo tháng (số dòng):")
print(so_b2b.loc[null_mask, "OrderedTimestamp"].dt.tz_localize(None).dt.to_period("M").value_counts().sort_index().to_dict())


Theo trạng thái: {'Done': 2944101, 'Canceled': 643490}
Tấn đặt / tấn giao: 1628.1 / 1364.1
Theo tháng (số dòng):
{Period('2025-01', 'M'): 305608, Period('2025-02', 'M'): 295829, Period('2025-03', 'M'): 477840, Period('2025-04', 'M'): 431478, Period('2025-05', 'M'): 348806, Period('2025-06', 'M'): 347187, Period('2025-07', 'M'): 373412, Period('2025-08', 'M'): 313237, Period('2025-09', 'M'): 230321, Period('2025-10', 'M'): 237048, Period('2025-11', 'M'): 226825}


In [41]:
# ===== 2.5f DEMAND THEO LƯỢNG ĐẶT (Done + Canceled hủy hẳn, đã bỏ trùng) =====
KEY = ["SOLineID", "ShipToID", "ProductID"]                          # khóa 1 dòng đơn thật
co_ma = so_b2b["SOLineID"] != "NULL"                                 # dòng có mã dòng đơn -> bỏ trùng được
thang = so_b2b["OrderedTimestamp"].dt.tz_localize(None).dt.to_period("M")

# (1) Dòng có mã: mỗi dòng đơn chỉ tính lượng đặt 1 lần (lấy lớn nhất giữa các phiếu Canceled / Done)
co = so_b2b[co_ma].assign(Thang=thang[co_ma])
dat_co_ma = co.groupby(KEY, observed=True).agg(Thang=("Thang", "first"), kg=("QtyOrderedInKG", "max"))

# (2) Dòng NULL: không bỏ trùng được -> tạm tính Done theo lượng đặt, Canceled tách riêng để xem
nul = so_b2b[~co_ma].assign(Thang=thang[~co_ma])
dat_null_done = nul[nul["OrderStatus"] == "Done"].groupby("Thang")["QtyOrderedInKG"].sum()
dat_null_canc = nul[nul["OrderStatus"] == "Canceled"].groupby("Thang")["QtyOrderedInKG"].sum()

bang = pd.DataFrame({
    "Dat_co_ma_botrung": dat_co_ma.groupby("Thang")["kg"].sum(),        # đơn có mã, đã bỏ trùng
    "Dat_NULL_Done": dat_null_done,                                     # đơn NULL đã giao
    "Dat_NULL_Canceled": dat_null_canc,                                 # đơn NULL bị hủy (không rõ có trùng không)
}).fillna(0) / 1000
bang["Demand_dat (khong NULL-Canceled)"] = bang["Dat_co_ma_botrung"] + bang["Dat_NULL_Done"]
bang["Demand_dat (co NULL-Canceled)"] = bang["Demand_dat (khong NULL-Canceled)"] + bang["Dat_NULL_Canceled"]
bang.loc["TONG"] = bang.sum()
print(bang.round(1))


         Dat_co_ma_botrung  Dat_NULL_Done  Dat_NULL_Canceled  Demand_dat (khong NULL-Canceled)  Demand_dat (co NULL-Canceled)
Thang                                                                                                                        
2025-01             1982.0          117.8               21.7                            2099.8                         2121.6
2025-02             2028.6          127.4               14.4                            2156.1                         2170.5
2025-03             2433.0          195.0               43.9                            2628.0                         2671.9
2025-04             2499.5          158.9               56.5                            2658.5                         2715.0
2025-05             2605.1          147.6               33.8                            2752.7                         2786.5
2025-06             2903.6          134.6               27.6                            3038.1                        

In [44]:
# ===== 2.5g CLEAN + XUẤT SALES ORDER B2B (chỉ Indonesia) =====
truoc_dong, truoc_kg = len(so_b2b), so_b2b["QtyOrderedInKG"].sum()

# (0) Chỉ giữ đơn của khách Indonesia (Decision_Log 2.1)
khach_id = set(pd.read_parquet(Path("../Model - Data used") / "customer_master.parquet")["LocationID"])
so = so_b2b[so_b2b["ShipToID"].astype(str).isin(khach_id)]

# (1) Bỏ dòng số lượng <= 0 và dòng trùng hoàn toàn (5.3)
so = so[so["QtyOrderedInKG"] > 0].drop_duplicates()

# (2) Bỏ trùng đơn (5.1): mỗi dòng đơn giữ dòng có lượng đặt lớn nhất; đơn mã NULL chỉ giữ Done
KEY = ["SOLineID", "ShipToID", "ProductID"]
co_ma = so["SOLineID"] != "NULL"
co = so[co_ma].sort_values("QtyOrderedInKG", ascending=False).drop_duplicates(KEY, keep="first")
nul = so[~co_ma & (so["OrderStatus"] == "Done")]
so = pd.concat([co, nul], ignore_index=True)

# (3) Bỏ 514 sản phẩm lỗi KGPerPallet (5.4)
pm_sach = set(pd.read_parquet(Path("../Model - Data used") / "product_master.parquet")["ProductID"])
so = so[so["ProductID"].astype(str).isin(pm_sach)]

print(f"Số dòng: {truoc_dong:,} -> {len(so):,}")
print(f"Tấn đặt: {truoc_kg/1000:,.0f} -> {so['QtyOrderedInKG'].sum()/1000:,.0f}")
print("Country còn lại:", so["Country"].value_counts().to_dict())          # kỳ vọng chỉ Indonesia

# Kiểm tra tự động
assert (so["QtyOrderedInKG"] > 0).all()
assert not so[so["SOLineID"] != "NULL"].duplicated(KEY).any()
assert so["ProductID"].astype(str).isin(pm_sach).all()
assert so["ShipToID"].astype(str).isin(khach_id).all()

out_file = Path("../Model - Data used") / "so_b2b.parquet"
so.to_parquet(out_file, index=False)
print(f"Đã lưu: {out_file} | {len(so):,} dòng")


Số dòng: 25,260,049 -> 24,084,062
Tấn đặt: 29,509 -> 26,790
Country còn lại: {'Indonesia': 24084062, 'Malaysia': 0}
Đã lưu: ..\Model - Data used\so_b2b.parquet | 24,084,062 dòng


### 2.6 Sales order B2C

### 2.7 Cost — Supply (GF)

### 2.8 Cost — Transfer (GF)

### 2.9 Cost — Last-mile B2B (GF, Unconstrained)

In [38]:
# ===== 2.9a LAST-MILE B2B: các file theo giá năm nào? =====
# Ý tưởng: trên cùng 1 tuyến (kho -> group), cost 2030 / cost 2025 ≈ 1.328 (hệ số trượt giá last-mile của công ty)
bl = cost_lastmile_b2b_bl.rename(columns={"CostPerPallet": "BL"})     # giá 2025, chỉ tuyến đang chạy
gf = cost_lastmile_b2b_gf.rename(columns={"CostPerPallet": "GF"})     # nghi giá 2030, mọi tuyến có thể chạy
unc = cost_lastmile_b2b_unc.rename(columns={"CostPerPallet": "Unc"})  # file ở root, chưa rõ năm

# Ghép từng cặp bảng theo tuyến; how="inner": chỉ giữ tuyến có ở cả 2 bảng
for ten, a, b in [("GF / BL", gf, bl), ("Unc / BL", unc, bl), ("Unc / GF", unc, gf)]:
    chung = a.merge(b, on=["OriginID", "DestinationID"], how="inner")
    ty_le = chung.iloc[:, 2] / chung.iloc[:, 3]                         # cost bảng thứ nhất / cost bảng thứ hai
    print(f"{ten}: {len(chung):,} tuyến chung | tỷ lệ trung vị = {ty_le.median():.3f} "
          f"| 5%–95%: {ty_le.quantile(.05):.3f}–{ty_le.quantile(.95):.3f}")

# Cách đọc:
#   GF / BL  ≈ 1.328 và khoảng 5–95% hẹp -> GF là giá 2030, chia 1.328 là ra giá 2025 cho mọi tuyến
#   Unc / BL ≈ 1.328 -> file root cũng giá 2030;  ≈ 1 -> file root là giá 2025
#   Unc / GF ≈ 1     -> 2 file cùng năm, chỉ khác số tuyến

GF / BL: 3,549 tuyến chung | tỷ lệ trung vị = 1.328 | 5%–95%: 1.328–1.328
Unc / BL: 3,549 tuyến chung | tỷ lệ trung vị = 1.328 | 5%–95%: 1.328–1.328
Unc / GF: 16,748 tuyến chung | tỷ lệ trung vị = 1.000 | 5%–95%: 1.000–1.000


In [39]:
# ===== 2.9b TUYẾN CHỈ CÓ Ở 1 FILE: đi từ kho nào? =====
key = ["OriginID", "DestinationID"]
chi_unc = unc.merge(gf[key], on=key, how="left", indicator=True).query("_merge == 'left_only'")  # chỉ có ở Unc
chi_gf = gf.merge(unc[key], on=key, how="left", indicator=True).query("_merge == 'left_only'")   # chỉ có ở GF

# Gắn loại kho / trạng thái của kho xuất phát (lấy từ facility master GF)
info = facility_master_gf.set_index("LocationID")[["FacilityType", "Status", "MainIsland"]]
for ten, df in [("Chỉ có ở Unc", chi_unc), ("Chỉ có ở GF", chi_gf)]:
    df = df.join(info, on="OriginID")                                      # thêm cột loại kho của Origin
    print(f"\n{ten}: {len(df):,} tuyến, từ {df['OriginID'].nunique()} kho")
    print(df.groupby(["Status", "FacilityType"], dropna=False).size())     # Non-existent = site greenfield



Chỉ có ở Unc: 6,104 tuyến, từ 29 kho
Status  FacilityType
Active  DC Direct        638
        DC Satellite     475
        DEPO            4460
        RDC              531
dtype: int64

Chỉ có ở GF: 74 tuyến, từ 11 kho
Status        FacilityType
Active        DC Satellite    15
Non-existent  Unknown         59
dtype: int64


### 2.10 Cost — Last-mile B2C (GF)

### 2.11 DOS (GF)

### 2.12 Lead time

## 3. Baseline — forced flow (tham chiếu, so sánh với GF)

### 3.1 Facility Master (BL vs GF)

### 3.2 Demand allocation B2C (BL vs GF)

### 3.3 Cost — Supply (BL vs GF)

### 3.4 Cost — Transfer (BL vs GF)

### 3.5 Cost — Last-mile B2B (BL vs GF)

### 3.6 Cost — Last-mile B2C (BL vs GF)

## 4. Tổng kết